C13. Длина как перевёрнутая U, а не линейный медиатор (Wu 2025, Cuadron 2025).....


In [1]:
import sys, os

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/evaluation_robustness')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
A = load_arms()
M = meta()


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base


In [2]:
def protocol_words(arm):
    d = A[arm]
    for c in ('winner_protocol', 'baseline_protocol_anonymous'):
        if c in d.columns and d[c].notna().mean() > 0.5:
            return d[c].map(words)
    return pd.Series(index=d.index, dtype=float)
W = {}
rows = []
for arm in A:
    w = protocol_words(arm)
    if w.notna().mean() < 0.5:
        continue
    W[arm] = w
    s = js(A, arm)
    ix = w.dropna().index.intersection(s.index)
    rows.append({'плечо': arm, 'n': len(ix), 'медиана слов': float(w.loc[ix].median()), 'среднее': float(w.loc[ix].mean()), 'Q1': float(w.loc[ix].quantile(0.25)), 'Q3': float(w.loc[ix].quantile(0.75)), 'Judge': float(s.loc[ix].mean())})
LW = pd.DataFrame(rows).set_index('плечо').sort_values('медиана слов')
show(LW, 'Длина протокола по плечам')



Длина протокола по плечам
------------------------------------------------------------
                          n  медиана слов  среднее       Q1       Q3  Judge
плечо                                                                      
baseline gemini         229      211.0000 217.7817 185.0000 243.0000 0.4247
baseline deepseekv4pro  229      227.0000 233.6987 196.0000 261.0000 0.5601
baseline qwen           229      266.0000 276.6332 239.0000 301.0000 0.5622
baseline glm            229      284.0000 293.2358 234.0000 348.0000 0.5870
baseline gpt            229      285.0000 299.2707 250.0000 335.0000 0.6258
USC@3 gpt               229      289.0000 301.9694 261.0000 333.0000 0.6177
baseline kimi           221      301.0000 304.9593 264.0000 341.0000 0.5955
USC@5 gpt               229      304.0000 313.5022 269.0000 350.0000 0.6284
USC@7 gpt               229      308.0000 317.2140 272.0000 349.0000 0.6280
baseline deepseekv4     228      311.0000 319.6009 269.0000 356.2500 0.6401


In [3]:
parts = []
for arm, w in W.items():
    s = js(A, arm)
    ix = w.dropna().index.intersection(s.index)
    parts.append(pd.DataFrame({'y': s.loc[ix].to_numpy(), 'w': w.loc[ix].to_numpy(), 'arm': arm, 'case': ix}))
L = pd.concat(parts, ignore_index=True)
L = L[L['w'] > 0].copy()
L['lw'] = np.log(L['w'])
L['lw_c'] = L['lw'] - L['lw'].mean()
L['lw2'] = L['lw_c'] ** 2
D = pd.get_dummies(L['arm'], prefix='arm', drop_first=True).astype(float)
print('наблюдений:', len(L), ' плеч:', L['arm'].nunique())
X1 = sm.add_constant(pd.concat([L[['lw_c']], D], axis=1))
X2 = sm.add_constant(pd.concat([L[['lw_c', 'lw2']], D], axis=1))
g = L['case'].astype('category').cat.codes
m1 = sm.OLS(L['y'], X1).fit(cov_type='cluster', cov_kwds={'groups': g})
m2 = sm.OLS(L['y'], X2).fit(cov_type='cluster', cov_kwds={'groups': g})
out = pd.DataFrame({'линейная коэф': m1.params, 'p': m1.pvalues, 'квадратичная коэф': m2.params, 'p ': m2.pvalues})
show(out.loc[['const', 'lw_c'] + (['lw2'] if 'lw2' in out.index else [])], 'Эффект длины: линейная и квадратичная спецификации')
print()
print('R² линейная = %.4f,  квадратичная = %.4f' % (m1.rsquared, m2.rsquared))
print('тест значимости квадратичного члена: p = %s' % fmt_p(m2.pvalues.get('lw2', np.nan)))
if m2.params.get('lw2', 0) < 0:
    opt = -m2.params['lw_c'] / (2 * m2.params['lw2'])
    opt_w = float(np.exp(opt + L['lw'].mean()))
    print('оптимум (максимум параболы): %.0f слов' % opt_w)
    print('наблюдаемый диапазон длин  : %.0f ... %.0f слов' % (L['w'].quantile(0.01), L['w'].quantile(0.99)))
    print('оптимум ВНУТРИ диапазона   : %s' % ('да' if L['w'].quantile(0.01) < opt_w < L['w'].quantile(0.99) else 'НЕТ (экстраполяция)'))
else:
    opt_w = np.nan
    print('квадратичный член неотрицателен — перевёрнутой U нет')


наблюдений: 4113  плеч: 18

Эффект длины: линейная и квадратичная спецификации
------------------------------------------------------------
       линейная коэф      p  квадратичная коэф     p 
const         0.7369 0.0000             0.7495 0.0000
lw_c          0.1906 0.0000             0.1728 0.0000
lw2              NaN    NaN            -0.0696 0.0000

R² линейная = 0.3045,  квадратичная = 0.3088
тест значимости квадратичного члена: p = 1.4e-06
оптимум (максимум параболы): 1234 слов
наблюдаемый диапазон длин  : 165 ... 711 слов
оптимум ВНУТРИ диапазона   : НЕТ (экстраполяция)


In [4]:
L['dec'] = pd.qcut(L['w'], 10, labels=False, duplicates='drop')
dec = L.groupby('dec').agg(n=('y', 'size'), медиана_слов=('w', 'median'), Judge=('y', 'mean'), SE=('y', lambda s: se_clt(s)))
show(dec, 'Средний балл по децилям длины (все плечи вместе)')
peak = dec['Judge'].idxmax()
print('\nмаксимум в дециле %d (медиана %.0f слов)' % (peak, dec.loc[peak, 'медиана_слов']))
print('падение от пика к последнему децилю: %+.4f' % (dec['Judge'].iloc[-1] - dec['Judge'].max()))



Средний балл по децилям длины (все плечи вместе)
------------------------------------------------------------
       n  медиана_слов  Judge     SE
dec                                 
0    419      202.0000 0.4732 0.0096
1    410      245.0000 0.5500 0.0084
2    413      272.0000 0.5909 0.0088
3    407      302.0000 0.6240 0.0086
4    412      338.0000 0.6642 0.0093
5    418      379.0000 0.6907 0.0078
6    404      427.0000 0.7236 0.0078
7    415      475.0000 0.7439 0.0075
8    406      530.0000 0.7894 0.0066
9    409      619.0000 0.7970 0.0070

максимум в дециле 9 (медиана 619 слов)
падение от пика к последнему децилю: +0.0000


In [5]:
rows = []
for arm in ['P-base', 'baseline gpt', 'homog gpt', 'USC@7 gpt', 'P-strong']:
    sub = L[L['arm'] == arm]
    if len(sub) < 60:
        continue
    q = pd.qcut(sub['w'], 5, labels=False, duplicates='drop')
    means = sub.groupby(q)['y'].mean()
    x = sm.add_constant(pd.DataFrame({'lw_c': sub['lw'] - sub['lw'].mean(), 'lw2': (sub['lw'] - sub['lw'].mean()) ** 2}))
    mm = sm.OLS(sub['y'], x).fit(cov_type='HC3')
    b2 = mm.params['lw2']
    optw = float(np.exp(-mm.params['lw_c'] / (2 * b2) + sub['lw'].mean())) if b2 < 0 else np.nan
    r = {'плечо': arm, 'n': len(sub), 'β₂': b2, 'p(β₂)': mm.pvalues['lw2'], 'оптимум, слов': optw, 'медиана плеча': float(sub['w'].median())}
    for i, v in means.items():
        r['Q%d' % (i + 1)] = v
    rows.append(r)
Q = pd.DataFrame(rows).set_index('плечо')
show(Q, 'Форма зависимости внутри плеч (квинтили длины и парабола)')
print()
print('«медиана плеча» против «оптимум»: если медиана ПРАВЕЕ оптимума,')
print('плечо в среднем переобдумывает (H13.3).')



Форма зависимости внутри плеч (квинтили длины и парабола)
------------------------------------------------------------
                n      β₂  p(β₂)  оптимум, слов  медиана плеча     Q1     Q2     Q3     Q4     Q5
плечо                                                                                            
P-base        229  0.3493 0.1676            NaN       472.0000 0.7854 0.7683 0.7543 0.7996 0.8443
baseline gpt  229  0.0729 0.6790            NaN       285.0000 0.5649 0.6221 0.6641 0.5686 0.7120
homog gpt     229 -0.3985 0.2441       506.6848       471.0000 0.7447 0.7520 0.7575 0.8024 0.7372
USC@7 gpt     229 -0.1230 0.6632       665.8885       308.0000 0.5838 0.6119 0.6367 0.6273 0.6801
P-strong      229 -0.2711 0.0646       722.7339       547.0000 0.7478 0.7794 0.8325 0.8498 0.8239

«медиана плеча» против «оптимум»: если медиана ПРАВЕЕ оптимума,
плечо в среднем переобдумывает (H13.3).


In [6]:
ext = [a for a in ['baseline kimi', 'baseline deepseekv4pro'] if a in A]
if ext:
    diff = pd.concat([js(A, a).rename(a) for a in ext], axis=1).mean(axis=1)
    L['diff'] = L['case'].map(diff)
    sub = L.dropna(subset=['diff']).copy()
    Ds = pd.get_dummies(sub['arm'], prefix='arm', drop_first=True).astype(float)
    X3 = sm.add_constant(pd.concat([sub[['lw_c', 'lw2', 'diff']], Ds], axis=1))
    m3 = sm.OLS(sub['y'], X3).fit(cov_type='cluster', cov_kwds={'groups': sub['case'].astype('category').cat.codes})
    show(pd.DataFrame({'коэф': m3.params, 'SE': m3.bse, 'p': m3.pvalues}).loc[['const', 'lw_c', 'lw2', 'diff']], 'Квадратичная модель с контролем внешней трудности кейса')
    print()
    print('Если β₂ сохраняет знак и значимость после контроля трудности,')
    print('форма «перевёрнутая U» не сводится к тому, что длинные протоколы')
    print('пишутся на трудных кейсах.')



Квадратичная модель с контролем внешней трудности кейса
------------------------------------------------------------
         коэф     SE      p
const  0.4088 0.0203 0.0000
lw_c   0.1529 0.0194 0.0000
lw2   -0.0439 0.0117 0.0002
diff   0.5938 0.0322 0.0000

Если β₂ сохраняет знак и значимость после контроля трудности,
форма «перевёрнутая U» не сводится к тому, что длинные протоколы
пишутся на трудных кейсах.


In [7]:
pair = L[L['arm'].isin(['P-base', 'baseline gpt'])].copy()
pair['panel'] = (pair['arm'] == 'P-base').astype(float)
gg = pair['case'].astype('category').cat.codes
raw = sm.OLS(pair['y'], sm.add_constant(pair[['panel']])).fit(cov_type='cluster', cov_kwds={'groups': gg})
lin = sm.OLS(pair['y'], sm.add_constant(pair[['panel', 'lw_c']])).fit(cov_type='cluster', cov_kwds={'groups': gg})
qua = sm.OLS(pair['y'], sm.add_constant(pair[['panel', 'lw_c', 'lw2']])).fit(cov_type='cluster', cov_kwds={'groups': gg})
rows = [{'модель': 'без контроля длины', 'эффект панели': raw.params['panel'], 'p': raw.pvalues['panel'], 'доля объяснённого, %': 0.0}, {'модель': '+ линейный log(слов)', 'эффект панели': lin.params['panel'], 'p': lin.pvalues['panel'], 'доля объяснённого, %': 100 * (1 - lin.params['panel'] / raw.params['panel'])}, {'модель': '+ квадратичный', 'эффект панели': qua.params['panel'], 'p': qua.pvalues['panel'], 'доля объяснённого, %': 100 * (1 - qua.params['panel'] / raw.params['panel'])}]
show(pd.DataFrame(rows).set_index('модель'), 'Медиация длиной: P-base против baseline gpt')



Медиация длиной: P-base против baseline gpt


------------------------------------------------------------
                      эффект панели      p  доля объяснённого, %
модель                                                          
без контроля длины           0.1641 0.0000                0.0000
+ линейный log(слов)         0.0938 0.0000               42.8468
+ квадратичный               0.0940 0.0000               42.7488


In [8]:
save(LW.reset_index(), 'nonlinear_protocol_length_effects_lengths.csv')
save(dec.reset_index(), 'nonlinear_protocol_length_effects_deciles.csv')
save(Q.reset_index(), 'nonlinear_protocol_length_effects_within_arm.csv')


сохранено: out/nonlinear_protocol_length_effects_lengths.csv  (18 строк)
сохранено: out/nonlinear_protocol_length_effects_deciles.csv  (10 строк)
сохранено: out/nonlinear_protocol_length_effects_within_arm.csv  (5 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\evaluation_robustness\\results\\nonlinear_protocol_length_effects_within_arm.csv'